In [1]:
import geemap
import utils as F
import ee
import concurrent.futures
import glob
import rasterio
import geopandas as gpd
from rasterio.merge import merge
from shapely.geometry import mapping
from shapely.ops import unary_union, transform
import shapely
import concurrent.futures
import os

In [4]:
import ee
ee.Initialize(project="ee-103dg017")
ee.Authenticate()

True

In [5]:
def force_2d(geom):
    # 兼容 Shapely 1.x/2.x：把 (x,y,z) -> (x,y)
    return transform(lambda x, y, z=None: (x, y), geom)

def shapely_to_ee_geometry(shp_geom):
    # 1) 先强制 2D
    shp_geom = force_2d(shp_geom)
    # 2) 修复无效几何（自相交等）
    if not shp_geom.is_valid:
        shp_geom = shp_geom.buffer(0)
    geo_json = mapping(shp_geom)
    gtype = geo_json["type"]
    coords = geo_json["coordinates"]

    if gtype == "Polygon":
        return ee.Geometry.Polygon(coords)
    elif gtype == "MultiPolygon":
        return ee.Geometry.MultiPolygon(coords)
    else:
        raise ValueError(f"Unsupported geometry type: {gtype}")
    
def cal_cloud_per(img):
    
    cloud_img = img.select('cloud')
    Cloud_count = cloud_img.reduceRegion(
      
      reducer = ee.Reducer.sum(),
      geometry = roi,
      scale = 30,
      maxPixels = 1e13
    ).get('cloud')
  
    count = cloud_img.reduceRegion(
      
      reducer = ee.Reducer.count(),
      geometry = roi,
      scale = 30,
      maxPixels =  1e13
    ).get('cloud')
    
    cloud_per = ee.Number(Cloud_count).divide(count).multiply(100)
    
    return img.set('cloud_per', cloud_per)
  
def CLIP(img):
    return img.clip(roi)

In [ ]:
path = r"E:\GEEdownload\UK_TF_roi\MGRS_UK_FK.shp"
gdf = gpd.read_file(path)

target_region = 'Wash_bay'
# --- 选 Morecambe 区 ---
region_grid = gdf[(gdf['name_1'] == target_region)]
MGRS_list = region_grid['Name'].values
print(f' {len(MGRS_list)} MGRS_tile were exit! ','\n', f'MGRS_list: {MGRS_list}')

if region_grid.empty:
    raise ValueError("筛选结果为空：检查字段 'name_1' 和筛选值。")

#load edge roi
edge_path = f'E:\GEEdownload\TK-TF_edge_collect\{target_region}_edge.shp'
edge_gdf = gpd.read_file(edge_path)
edge_gdf  = geemap.geopandas_to_ee(edge_gdf)       
edge_roi = edge_gdf.geometry() 


In [6]:
target_region_list = ['Wash_bay', 'Henber', 'Thames_esturay', 'Region3b1', 'Solway', 
                      'Morecambe', 'ribble', 'Severn_Esturay', 'Region4b', 'Region4a4']
year_list = [2022, 2023]
target_region_list = ['Thames_esturay']

for j in range(len(target_region_list)):
    
    target_region = target_region_list[j]

    path = r"E:\GEEdownload\UK_TF_roi\MGRS_UK_FK.shp"
    gdf = gpd.read_file(path)

    # target_region = 'Region1a'

    region_grid = gdf[(gdf['name_1'] == target_region)]
    MGRS_list = region_grid['Name'].values
    print(f' {len(MGRS_list)} MGRS_tile were exit! ','\n', f'MGRS_list: {MGRS_list}')

    if region_grid.empty:
        raise ValueError("筛选结果为空：检查字段 'name_1' 和筛选值。")

    #load edge roi
    edge_path = f'E:\GEEdownload\TK-TF_edge_collect\{target_region}_edge.shp'
    edge_gdf = gpd.read_file(edge_path)
    edge_gdf  = geemap.geopandas_to_ee(edge_gdf)       
    edge_roi = edge_gdf.geometry() 

    for year in year_list:

        start_date, end_date = f'{year}-01-01', f'{year+1}-01-01'

        for i in range(len(MGRS_list)):

            MGRS_TILE = MGRS_list[i]
            cloud_limit = 40
            apply_MGRS_TILE = True
            
            #读取 region MGRS1
            grid = region_grid[(region_grid['Name'] == MGRS_TILE)]

            merged = unary_union([force_2d(geom) for geom in grid.geometry if geom is not None])
            roi = shapely_to_ee_geometry(merged)


            S2 = F.get_s2_sr_cld_col(roi, start_date, end_date, 100)

            # S2.size().getInfo()
            if apply_MGRS_TILE:
                print('--apply MGRS_TILE--')
                S2 = S2.filter(ee.Filter.eq('MGRS_TILE', MGRS_TILE))
            
            S2 = S2.map(F.VI_cal_S2).map(F.add_cloud_bands)\
                    .map(CLIP).map(cal_cloud_per)\
                    .filter(ee.Filter.gt('cloud_per', cloud_limit))

            L9_collect = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
            L8_collect = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
            L9 = F.Landsat_processing(roi, L9_collect, start_date, end_date).map(cal_cloud_per).filter(ee.Filter.gt('cloud_per', 40)).map(F.VI_cal_L8)
            L8 = F.Landsat_processing(roi, L8_collect, start_date, end_date).map(cal_cloud_per).filter(ee.Filter.gt('cloud_per', 40)).map(F.VI_cal_L8)

            all_TF_collect = S2.merge(L8).merge(L9)
            all_TF_collect = ee.ImageCollection(all_TF_collect)
            sum_TF = all_TF_collect.map(F.TF_binary_cal).toBands().reduce(ee.Reducer.sum())
            sum_cloud = all_TF_collect.select('cloud').toBands().reduce(ee.Reducer.sum())
            Frequency = sum_TF.divide(sum_cloud).rename('Freq')
            
            S2_2022 = S2.map(F.normalized_TFI).map(F.TF_get)
            Norm_S2_2022 = S2_2022.map(F.Norm)
            cloud_sum = S2_2022.select('cloud').toBands().reduce(ee.Reducer.sum())

            NDWI_norm_min = S2_2022.filter(ee.Filter.gt('cloud_per', 60)).select('NDWI_norm').reduce(ee.Reducer.percentile([1])).rename('NDWI_min')
            NDWI_mean = S2_2022.select('NDWI').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('mean_NDWI')
            MNDWI_mean = S2_2022.select('MNDWI').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('mean_MNDWI')
            composit_b8A = S2_2022.select('B8A').reduce(ee.Reducer.sum()).divide(cloud_sum).divide(10000).rename('mean_B8A')
            NDVI_mean = S2_2022.select('NDVI').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('mean_NDVI')
            S2_NDWI_min = S2_2022.select('NDWI').reduce(ee.Reducer.percentile([1])).rename('S2_NDWI_min')

            S2_NDWI_norm_min = S2_2022.filter(ee.Filter.gt('cloud_per', 60)).select('NDWI_norm').reduce(ee.Reducer.percentile([1])).rename('NDWI_min')
            TF_mask = S2_NDWI_min.lt(-0.1).And(Frequency.gt(0.05)).And(Frequency.lt(0.95)).clip(edge_roi).unmask(0).rename('mask')

            feature_bands = Frequency.addBands(NDWI_mean).addBands(composit_b8A).addBands(MNDWI_mean).addBands(NDVI_mean).addBands(S2_NDWI_min).addBands(TF_mask)
            
            coords = roi.bounds().coordinates().getInfo()[0]
            minLon, minLat = coords[0]
            maxLon, maxLat = coords[2]

            # 经纬度换算（粗略）
            tile_size_deg = 5000 / 111320

            tiles = []
            lat = minLat
            while lat < maxLat:
                lon = minLon
                while lon < maxLon:
                    lon2 = min(lon + tile_size_deg, maxLon)
                    lat2 = min(lat + tile_size_deg, maxLat)
                    tiles.append(ee.Geometry.BBox(lon, lat, lon2, lat2))
                    lon += tile_size_deg
                lat += tile_size_deg

            print(f"将切成 {len(tiles)} 个切片进行下载...")

            output_dir = f"E:\A_tile_from_GEE\{target_region}_{year}_Tile"
            os.makedirs(output_dir, exist_ok=True)

            def export_tile(i_tile):
                i, tile = i_tile
                filename = os.path.join(output_dir, f"{MGRS_TILE}_tile_{i+1}.tif")
                print(f"开始导出 {filename} ...")
                geemap.ee_export_image(
                    feature_bands,
                    filename=filename,
                    scale=30,
                    region=tile,
                    file_per_band=False
                )
                print(f"完成导出 {filename}")
                return filename

            # 用线程池并发执行导出
            with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
                results = list(executor.map(export_tile, enumerate(tiles)))
            print(f"✅{MGRS_TILE} 所有切片导出完成！")
            
        print("✅ 所有切片导出完成！")

        src_files_to_mosaic = [rasterio.open(fp) for fp in glob.glob(f"{output_dir}/*.tif")]
        mosaic, out_trans = merge(src_files_to_mosaic)

        out_meta = src_files_to_mosaic[0].meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": mosaic.shape[1],
            "width": mosaic.shape[2],
            "transform": out_trans
        })

        merged_path = f"E:\GEE_download\Feature_bands/{target_region}_{year}_FB6_new8.tif"
        with rasterio.open(merged_path, "w", **out_meta) as dest:
            dest.write(mosaic)

        print(f"✅ 已完成拼接，输出文件: {merged_path}")

 2 MGRS_tile were exit!  
 MGRS_list: ['31UCS' '31UCT']
--apply MGRS_TILE--
将切成 135 个切片进行下载...
开始导出 E:\A_tile_from_GEE\Thames_esturay_2022_Tile\31UCS_tile_1.tif ...
Generating URL ...
开始导出 E:\A_tile_from_GEE\Thames_esturay_2022_Tile\31UCS_tile_2.tif ...
Generating URL ...
开始导出 E:\A_tile_from_GEE\Thames_esturay_2022_Tile\31UCS_tile_3.tif ...
Generating URL ...
开始导出 E:\A_tile_from_GEE\Thames_esturay_2022_Tile\31UCS_tile_4.tif ...
Generating URL ...
开始导出 E:\A_tile_from_GEE\Thames_esturay_2022_Tile\31UCS_tile_5.tif ...
Generating URL ...
开始导出 E:\A_tile_from_GEE\Thames_esturay_2022_Tile\31UCS_tile_6.tif ...
Generating URL ...
开始导出 E:\A_tile_from_GEE\Thames_esturay_2022_Tile\31UCS_tile_7.tif ...
Generating URL ...
开始导出 E:\A_tile_from_GEE\Thames_esturay_2022_Tile\31UCS_tile_8.tif ...
Generating URL ...
开始导出 E:\A_tile_from_GEE\Thames_esturay_2022_Tile\31UCS_tile_9.tif ...
Generating URL ...
开始导出 E:\A_tile_from_GEE\Thames_esturay_2022_Tile\31UCS_tile_10.tif ...
Generating URL ...
开始导出 E:\A_tile

In [33]:
import glob
import rasterio
from rasterio.merge import merge

src_files_to_mosaic = [rasterio.open(fp) for fp in glob.glob(f"{output_dir}/*.tif")]
mosaic, out_trans = merge(src_files_to_mosaic)

out_meta = src_files_to_mosaic[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_trans
})

merged_path = f"E:\GEE_download/Region1a_FB_2024_2.tif"
with rasterio.open(merged_path, "w", **out_meta) as dest:
    dest.write(mosaic)

print(f"✅ 已完成拼接，输出文件: {merged_path}")

✅ 已完成拼接，输出文件: E:\GEE_download/Region1a_FB_2024_2.tif
